# LightGBM регрессия

In [1]:
import warnings
warnings.filterwarnings('ignore')

import os
import numpy as np
import pandas as pd

from sklearn.model_selection import cross_val_score, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler, MinMaxScaler, RobustScaler, FunctionTransformer, PolynomialFeatures
from sklearn.impute import SimpleImputer


def cv(model, X, y):
    scores = cross_val_score(model, X, y, cv=5, scoring='neg_mean_absolute_percentage_error', n_jobs=1)
    mape = -scores
    print(f"MAPE: {mape.mean():.6f} +- {mape.std():.6f}")
    return mape.mean()


def save_submission(model, X_train, y_train, X_test, ids, path):
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    os.makedirs('out', exist_ok=True)
    pd.DataFrame({'ID': ids, 'salary_mean_net': pred}).to_csv(path, index=False)
    print('saved to', path)


log_transform = FunctionTransformer(lambda a: np.log1p(np.clip(a, 0, None)), feature_names_out='one-to-one')

from lightgbm import LGBMRegressor


## Подготовка данных

In [2]:
df_train = pd.read_csv('data/train.csv')
X = df_train.iloc[:, :-1].copy()
y = df_train.iloc[:, -1].copy()

X[['unified_address_city', 'unified_address_region']] = X[['unified_address_city', 'unified_address_region']].fillna('missing')
for col in ['key_skills_name', 'languages_name', 'employer_industries']:
    if col in X.columns:
        X[col] = X[col].fillna('missing')

X = X.drop(columns=['id','employer_id','raw_description','raw_branded_description','lemmaized_wo_stopwords_raw_description','lemmaized_wo_stopwords_raw_branded_description','name','unified_address_country'], errors='ignore').copy()

counts = X['employer_name'].value_counts()
rare_categories = counts[counts < 200].index
X['employer_name'] = X['employer_name'].replace(rare_categories, 'Other')

cat_cols = X.select_dtypes(include=['object']).columns.tolist()
num_cols = [c for c in X.columns if c not in cat_cols]


## 1. Выбор скейлера

In [3]:
def build_preprocessor(scaler_name, feature_mode='none'):
    if scaler_name == 'standard':
        scaler = StandardScaler()
    elif scaler_name == 'minmax':
        scaler = MinMaxScaler()
    else:
        scaler = RobustScaler()

    num_steps = [('imp', SimpleImputer(strategy='median'))]

    if feature_mode in ['log', 'log_poly']:
        num_steps.append(('log', log_transform))

    num_steps.append(('sc', scaler))

    if feature_mode in ['poly', 'log_poly']:
        num_steps.append(('poly', PolynomialFeatures(degree=2, include_bias=False, interaction_only=True)))

    pre = ColumnTransformer([
        ('num', Pipeline(num_steps), num_cols),
        ('cat', Pipeline([
            ('imp', SimpleImputer(strategy='most_frequent')),
            ('enc', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)),
        ]), cat_cols),
    ])
    return pre


### Эксперимент 1.1: StandardScaler

In [4]:
pre_std = build_preprocessor('standard', 'none')
pipe_std = Pipeline([('preprocessor', pre_std), ('model', LGBMRegressor(objective='regression_l1', n_estimators=1200, learning_rate=0.03, num_leaves=63, n_jobs=1))])
mape_std = cv(pipe_std, X, y)


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002515 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1246
[LightGBM] [Info] Number of data points in the train set: 39240, number of used features: 17
[LightGBM] [Info] Start training from score 42630.000000
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002842 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1242
[LightGBM] [Info] Number of data points in the train set: 39241, number of used features: 17
[LightGBM] [Info] Start training from score 43000.000000
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002564 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is n

### Эксперимент 1.2: MinMaxScaler

In [5]:
pre_mm = build_preprocessor('minmax', 'none')
pipe_mm = Pipeline([('preprocessor', pre_mm), ('model', LGBMRegressor(objective='regression_l1', n_estimators=1200, learning_rate=0.03, num_leaves=63, n_jobs=1))])
mape_mm = cv(pipe_mm, X, y)


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002868 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1244
[LightGBM] [Info] Number of data points in the train set: 39240, number of used features: 17
[LightGBM] [Info] Start training from score 42630.000000
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002641 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1240
[LightGBM] [Info] Number of data points in the train set: 39241, number of used features: 17
[LightGBM] [Info] Start training from score 43000.000000
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002561 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is n

### Эксперимент 1.3: RobustScaler

In [6]:
pre_rb = build_preprocessor('robust', 'none')
pipe_rb = Pipeline([('preprocessor', pre_rb), ('model', LGBMRegressor(objective='regression_l1', n_estimators=1200, learning_rate=0.03, num_leaves=63, n_jobs=1))])
mape_rb = cv(pipe_rb, X, y)


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002593 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1244
[LightGBM] [Info] Number of data points in the train set: 39240, number of used features: 17
[LightGBM] [Info] Start training from score 42630.000000
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002555 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1240
[LightGBM] [Info] Number of data points in the train set: 39241, number of used features: 17
[LightGBM] [Info] Start training from score 43000.000000
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002524 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is n

In [7]:
scaler_scores = {'standard': mape_std, 'minmax': mape_mm, 'robust': mape_rb}
best_scaler = min(scaler_scores, key=scaler_scores.get)
print('Best scaler:', best_scaler, 'MAPE=', scaler_scores[best_scaler])


Best scaler: standard MAPE= 0.346613053355693


## 2. На лучшем скейлере: без / log / poly / log+poly

### Эксперимент 2.1: без дополнительных фич

In [8]:
pre_none = build_preprocessor(best_scaler, 'none')
pipe_none = Pipeline([('preprocessor', pre_none), ('model', LGBMRegressor(objective='regression_l1', n_estimators=1200, learning_rate=0.03, num_leaves=63, n_jobs=1))])
mape_none = cv(pipe_none, X, y)


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002915 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1246
[LightGBM] [Info] Number of data points in the train set: 39240, number of used features: 17
[LightGBM] [Info] Start training from score 42630.000000
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002712 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1242
[LightGBM] [Info] Number of data points in the train set: 39241, number of used features: 17
[LightGBM] [Info] Start training from score 43000.000000
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002477 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is n

### Эксперимент 2.2: логарифмические фичи

In [9]:
pre_log = build_preprocessor(best_scaler, 'log')
pipe_log = Pipeline([('preprocessor', pre_log), ('model', LGBMRegressor(objective='regression_l1', n_estimators=1200, learning_rate=0.03, num_leaves=63, n_jobs=1))])
mape_log = cv(pipe_log, X, y)


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002710 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1246
[LightGBM] [Info] Number of data points in the train set: 39240, number of used features: 17
[LightGBM] [Info] Start training from score 42630.000000
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.006616 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1242
[LightGBM] [Info] Number of data points in the train set: 39241, number of used features: 17
[LightGBM] [Info] Start training from score 43000.000000
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001750 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] 

### Эксперимент 2.3: полиномиальные фичи

In [10]:
pre_poly = build_preprocessor(best_scaler, 'poly')
pipe_poly = Pipeline([('preprocessor', pre_poly), ('model', LGBMRegressor(objective='regression_l1', n_estimators=1200, learning_rate=0.03, num_leaves=63, n_jobs=1))])
mape_poly = cv(pipe_poly, X, y)


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002860 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1251
[LightGBM] [Info] Number of data points in the train set: 39240, number of used features: 18
[LightGBM] [Info] Start training from score 42630.000000
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.003226 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1247
[LightGBM] [Info] Number of data points in the train set: 39241, number of used features: 18
[LightGBM] [Info] Start training from score 43000.000000
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002735 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is n

### Эксперимент 2.4: логарифмические + полиномиальные фичи

In [11]:
pre_log_poly = build_preprocessor(best_scaler, 'log_poly')
pipe_log_poly = Pipeline([('preprocessor', pre_log_poly), ('model', LGBMRegressor(objective='regression_l1', n_estimators=1200, learning_rate=0.03, num_leaves=63, n_jobs=1))])
mape_log_poly = cv(pipe_log_poly, X, y)


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002550 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1251
[LightGBM] [Info] Number of data points in the train set: 39240, number of used features: 18
[LightGBM] [Info] Start training from score 42630.000000
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002470 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1247
[LightGBM] [Info] Number of data points in the train set: 39241, number of used features: 18
[LightGBM] [Info] Start training from score 43000.000000
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002729 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is n

In [12]:
feature_scores = {'none': mape_none, 'log': mape_log, 'poly': mape_poly, 'log_poly': mape_log_poly}
pre_map = {'none': pre_none, 'log': pre_log, 'poly': pre_poly, 'log_poly': pre_log_poly}
best_feature_mode = min(feature_scores, key=feature_scores.get)
best_pre = pre_map[best_feature_mode]
print('Best feature mode:', best_feature_mode, 'MAPE=', feature_scores[best_feature_mode])


Best feature mode: poly MAPE= 0.3449814252341513


## 3. Подбор параметров на лучшем варианте

In [13]:
grid = GridSearchCV(
    estimator=Pipeline([('preprocessor', best_pre), ('model', LGBMRegressor())]),
    param_grid={'model__n_estimators':[600,1000,1400],'model__learning_rate':[0.02,0.03,0.05],'model__num_leaves':[31,63,127]},
    scoring='neg_mean_absolute_percentage_error',
    cv=5,
    n_jobs=1,
    verbose=1,
)

grid.fit(X, y)
print('Best params:', grid.best_params_)
print('Best CV MAPE:', -grid.best_score_)
best_model = grid.best_estimator_


Fitting 5 folds for each of 27 candidates, totalling 135 fits
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.002026 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1251
[LightGBM] [Info] Number of data points in the train set: 39240, number of used features: 18
[LightGBM] [Info] Start training from score 51057.478374
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001706 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1247
[LightGBM] [Info] Number of data points in the train set: 39241, number of used features: 18
[LightGBM] [Info] Start training from score 51130.235936
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000906 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] To

## Предикт теста и сохранение

In [15]:
df_test = pd.read_csv('data/test_x.csv')
X_test = df_test.copy()
X_test[['unified_address_city', 'unified_address_region']] = X_test[['unified_address_city', 'unified_address_region']].fillna('missing')
for col in ['key_skills_name', 'languages_name', 'employer_industries']:
    if col in X_test.columns:
        X_test[col] = X_test[col].fillna('missing')
X_test = X_test.drop(columns=['id','employer_id','raw_description','raw_branded_description','lemmaized_wo_stopwords_raw_description','lemmaized_wo_stopwords_raw_branded_description','name','unified_address_country'], errors='ignore').copy()
X_test['employer_name'] = X_test['employer_name'].replace(rare_categories, 'Other')

save_submission(best_model, X, y, X_test, df_test['id'], 'out/lightgbm_submission.csv')


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000895 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1262
[LightGBM] [Info] Number of data points in the train set: 49051, number of used features: 18
[LightGBM] [Info] Start training from score 51094.688582
saved to out/lightgbm_submission.csv
